In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd

# ---- Fix TF 2.18 + protobuf 6 crash:
# Force pure-Python protobuf runtime and API implementation version BEFORE importing TF.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf

keras = tf.keras

print("TF version:", tf.__version__)
try:
    print("Keras version:", keras.__version__)
except Exception:
    print("Keras version: (unavailable)")

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

try:
    tf.config.experimental.enable_op_determinism(True)
except Exception:
    pass

try:
    tf.config.threading.set_intra_op_parallelism_threads(0)
    tf.config.threading.set_inter_op_parallelism_threads(0)
except Exception:
    pass

try:
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception:
    pass




2026-05-12 17:16:48.898135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778606208.911241      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778606208.915304      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0
Keras version: 3.8.0


In [2]:
def _first_existing_path(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


output_dir = "./"

train_csv_path = _first_existing_path(
    [
        "../input/plant-pathology-2021-fgvc8/train.csv",
        "/kaggle/input/plant-pathology-2021-fgvc8/train.csv",
        "../input/train.csv",
        "/kaggle/input/train.csv",
    ]
)
test_dir = _first_existing_path(
    [
        "../input/plant-pathology-2021-fgvc8/test_images/",
        "/kaggle/input/plant-pathology-2021-fgvc8/test_images/",
        "../input/test_images/",
        "/kaggle/input/test_images/",
    ]
)
train_dir = _first_existing_path(
    [
        "../input/plant-pathology-2021-fgvc8/train_images/",
        "/kaggle/input/plant-pathology-2021-fgvc8/train_images/",
        "../input/train_images/",
        "/kaggle/input/train_images/",
    ]
)

model_dir = "../input/dense-e1/epoch-1"

if train_csv_path is None:
    raise FileNotFoundError(
        "Could not locate train.csv in expected Kaggle input paths."
    )
if test_dir is None or not os.path.isdir(test_dir):
    raise FileNotFoundError(
        "Could not locate test_images/ directory in expected Kaggle input paths."
    )
if train_dir is None or not os.path.isdir(train_dir):
    raise FileNotFoundError(
        "Could not locate train_images/ directory in expected Kaggle input paths."
    )

image_dims = (300, 300, 3)
data_set = pd.read_csv(train_csv_path)
df_labels = data_set["labels"]
one_hot = df_labels.str.get_dummies(sep=" ")
dataset_labels = one_hot.columns.to_list()
num_classes = len(dataset_labels)

print("Train CSV:", train_csv_path)
print("Train dir:", train_dir)
print("Test dir :", test_dir)
print("Num classes:", num_classes)
print("First classes:", dataset_labels[:10])




Train CSV: ../input/plant-pathology-2021-fgvc8/train.csv
Train dir: ../input/plant-pathology-2021-fgvc8/train_images/
Test dir : ../input/plant-pathology-2021-fgvc8/test_images/
Num classes: 6
First classes: ['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab']


In [3]:
if __name__ == "__main__":

    @tf.function(reduce_retracing=True)
    def _decode_and_resize_jpeg(img_path: tf.Tensor) -> tf.Tensor:
        bytes_ = tf.io.read_file(img_path)
        img = tf.io.decode_jpeg(bytes_, channels=3, dct_method="INTEGER_FAST")
        img = tf.image.convert_image_dtype(img, tf.float32)  # [0,1]
        img = tf.image.resize(img, [image_dims[0], image_dims[1]], method="bilinear")
        img = img * 255.0  # keep original inference scaling
        return img

    def _build_infer_model_from_savedmodel(saved_model_dir: str) -> keras.Model:
        endpoints_to_try = ["serving_default", "serve", "call"]
        last_err = None
        for endpoint in endpoints_to_try:
            try:
                layer = keras.layers.TFSMLayer(saved_model_dir, call_endpoint=endpoint)
                inp = keras.Input(shape=image_dims, dtype=tf.float32, name="image")
                out = layer(inp)
                if isinstance(out, dict):
                    for k in ["outputs", "predictions", "logits", "output_0"]:
                        if k in out:
                            out = out[k]
                            break
                    if isinstance(out, dict):
                        out = list(out.values())[0]
                return keras.Model(inp, out)
            except Exception as e:
                last_err = e
        raise RuntimeError(
            f"Could not load SavedModel from {saved_model_dir}. Last error: {last_err}"
        )

    def _savedmodel_exists(d: str) -> bool:
        return os.path.exists(os.path.join(d, "saved_model.pb")) or os.path.exists(
            os.path.join(d, "saved_model.pbtxt")
        )

    candidate_model_dirs = [
        model_dir,
        "/kaggle/input/dense-e1/epoch-1",
        "../input/dense-e1/epoch-1",
        "/kaggle/input/dense-e1",
        "../input/dense-e1",
        "/kaggle/input/dense-e1/epoch-1/epoch-1",
        "../input/dense-e1/epoch-1/epoch-1",
    ]
    model_dir_resolved = None
    for d in candidate_model_dirs:
        if d and os.path.isdir(d) and _savedmodel_exists(d):
            model_dir_resolved = d
            break

    def _build_fallback_train_model() -> keras.Model:
        inputs = keras.Input(shape=image_dims, dtype=tf.float32, name="image")
        x = inputs / 255.0
        x = keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
        x = keras.layers.MaxPooling2D()(x)
        x = keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
        x = keras.layers.MaxPooling2D()(x)
        x = keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
        x = keras.layers.GlobalAveragePooling2D()(x)
        x = keras.layers.Dropout(0.2)(x)
        outputs = keras.layers.Dense(num_classes, activation="sigmoid")(x)
        model = keras.Model(inputs, outputs)
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
        )
        return model

    cache_dir = os.path.join(output_dir, "tfdata_cache")
    os.makedirs(cache_dir, exist_ok=True)

    # Will be populated only when we have a validation split (fallback-training path)
    valid_ds = None
    va_y = None

    if model_dir_resolved is not None:
        print(f"Loading SavedModel from: {model_dir_resolved}")
        model = _build_infer_model_from_savedmodel(model_dir_resolved)
    else:
        print(
            f"SavedModel not found at {model_dir}. Training a fallback model from train_images/..."
        )
        model = _build_fallback_train_model()

        label_matrix = one_hot.values.astype("float32", copy=False)

        img_paths = np.array(
            [os.path.join(train_dir, f) for f in data_set["image"].values], dtype=object
        )

        rng = np.random.RandomState(0)
        idx = np.arange(len(img_paths))
        rng.shuffle(idx)
        split = int(0.95 * len(idx))
        tr_idx, va_idx = idx[:split], idx[split:]

        tr_paths, tr_y = img_paths[tr_idx], label_matrix[tr_idx]
        va_paths, va_y = img_paths[va_idx], label_matrix[va_idx]

        base_opts = tf.data.Options()
        base_opts.experimental_deterministic = True

        def _make_ds(paths, y, training: bool, cache_name: str):
            ds = tf.data.Dataset.from_tensor_slices((paths, y)).with_options(base_opts)

            opts = tf.data.Options()
            opts.experimental_deterministic = True
            try:
                opts.experimental_optimization.map_parallelization = True
                opts.experimental_optimization.parallel_batch = True
            except Exception:
                pass
            ds = ds.with_options(opts)

            ds = ds.map(
                lambda p, lab: (_decode_and_resize_jpeg(p), lab),
                num_parallel_calls=tf.data.AUTOTUNE,
                deterministic=True,
            )
            ds = ds.cache(os.path.join(cache_dir, cache_name))
            if training:
                ds = ds.shuffle(
                    buffer_size=min(8192, int(paths.shape[0])),
                    seed=0,
                    reshuffle_each_iteration=True,
                )
            ds = ds.batch(32, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
            return ds

        train_ds = _make_ds(tr_paths, tr_y, training=True, cache_name="train.cache")
        valid_ds = _make_ds(va_paths, va_y, training=False, cache_name="valid.cache")

        model.fit(train_ds, validation_data=valid_ds, epochs=2, verbose=2)

    # ---- Threshold calibration for mean-F1 (score move toward target), but only when we have val labels.
    # This does NOT change model architecture/training; it only chooses a better global cutoff.
    threshold = 0.5
    if valid_ds is not None and va_y is not None:
        try:
            val_preds = model.predict(valid_ds, verbose=0)
            val_preds = np.asarray(val_preds)
            if val_preds.ndim == 1:
                val_preds = val_preds[:, None]

            y_true = np.asarray(va_y)
            if y_true.ndim == 1:
                y_true = y_true[:, None]

            def _macro_f1(y_t: np.ndarray, y_p_bin: np.ndarray) -> float:
                eps = 1e-7
                tp = (y_t * y_p_bin).sum(axis=0)
                fp = ((1.0 - y_t) * y_p_bin).sum(axis=0)
                fn = (y_t * (1.0 - y_p_bin)).sum(axis=0)
                f1 = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
                return float(np.mean(f1))

            thr_grid = np.arange(0.10, 0.901, 0.05, dtype=np.float32)
            best_thr = threshold
            best_f1 = -1.0
            for thr in thr_grid:
                y_pred_bin = (val_preds > float(thr)).astype(np.float32)
                f1 = _macro_f1(y_true, y_pred_bin)
                if f1 > best_f1:
                    best_f1 = f1
                    best_thr = float(thr)

            threshold = best_thr
            print(
                f"Calibrated global threshold on validation: {threshold:.2f} (val macro-F1={best_f1:.4f})"
            )
        except Exception as e:
            print("Threshold calibration skipped due to error:", repr(e))
            threshold = 0.5

    try:
        listing = tf.io.gfile.listdir(test_dir)
    except Exception:
        listing = os.listdir(test_dir)
    images_path_list = sorted([f for f in listing if f[-4:].lower() == ".jpg"])
    if len(images_path_list) == 0:
        raise RuntimeError(f"No .jpg files found in test_dir={test_dir}")

    test_paths = np.array(
        [os.path.join(test_dir, f) for f in images_path_list], dtype=object
    )
    test_names = np.array(images_path_list, dtype=object)

    base_opts = tf.data.Options()
    base_opts.experimental_deterministic = True

    def _make_test_ds(paths):
        ds = tf.data.Dataset.from_tensor_slices(paths).with_options(base_opts)

        opts = tf.data.Options()
        opts.experimental_deterministic = True
        try:
            opts.experimental_optimization.map_parallelization = True
            opts.experimental_optimization.parallel_batch = True
        except Exception:
            pass
        ds = ds.with_options(opts)

        ds = ds.map(
            _decode_and_resize_jpeg,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=True,
        )
        ds = ds.batch(64, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
        return ds

    test_ds = _make_test_ds(test_paths)

    preds = model.predict(test_ds, verbose=0)
    preds = np.asarray(preds)
    if preds.ndim == 1:
        preds = preds[:, None]

    mask = preds > threshold

    if "healthy" in dataset_labels:
        healthy_idx = dataset_labels.index("healthy")
        any_other = np.any(np.delete(mask, healthy_idx, axis=1), axis=1)
        mask[any_other, healthy_idx] = False

    cls_idx_to_str = np.array(dataset_labels, dtype=object)

    def _row_to_labels(row_mask: np.ndarray) -> str:
        idxs = np.flatnonzero(row_mask)
        if idxs.size == 0:
            return "healthy"
        return " ".join(cls_idx_to_str[idxs].tolist()).strip()

    labels_out = [_row_to_labels(row) for row in mask]

    csv_pd = pd.DataFrame({"image": test_names, "labels": labels_out})
    sub_path = os.path.join(output_dir, "submission.csv")
    csv_pd.to_csv(sub_path, index=False)
    print("Wrote submission.csv with shape:", csv_pd.shape, "to", sub_path)
    print(csv_pd.head())

SavedModel not found at ../input/dense-e1/epoch-1. Training a fallback model from train_images/...


I0000 00:00:1778606216.103516      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:86:00.0, compute capability: 8.6


Epoch 1/2


I0000 00:00:1778606452.666563     108 service.cc:148] XLA service 0x7f3fe4009f10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778606452.666611     108 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1778606452.964565     108 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778606456.561254     108 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


443/443 - 444s - 1s/step - loss: 0.4370 - val_loss: 0.4114
Epoch 2/2
443/443 - 20s - 45ms/step - loss: 0.4099 - val_loss: 0.4157
Calibrated global threshold on validation: 0.25 (val macro-F1=0.4234)
Wrote submission.csv with shape: (3727, 2) to ./submission.csv
                  image                                          labels
0  801f78399a44e7af.jpg  complex frog_eye_leaf_spot powdery_mildew scab
1  8023c3f31f875b6c.jpg                                         healthy
2  802969daaddbbc8c.jpg                 complex frog_eye_leaf_spot scab
3  802b59956a7aa5e7.jpg                                         healthy
4  803f621b4757e0af.jpg                         frog_eye_leaf_spot scab
